### Geok Compressed data XGboost training and testing

In [ ]:
import os
import json
from datetime import datetime, timedelta
from pathlib import Path
from typing import Tuple, Optional, Union, List, Dict, Any
from dataclasses import dataclass, field
import multiprocessing
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import skew, kurtosis, entropy, randint, uniform, loguniform
from scipy.fft import fft
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.calibration import calibration_curve
import xgboost as xgb
from xgboost import plot_importance
import joblib
from joblib import Parallel, delayed
import pyarrow as pa
from tqdm import tqdm
import mlflow
from mlflow.models.signature import infer_signature
from joblib import load
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_curve, auc,
    precision_recall_curve, average_precision_score, confusion_matrix, classification_report
)
from sklearn.utils import resample

In [ ]:
time_stamp = datetime.now().strftime("%d%m%y_%H%M")

In [ ]:
#ENVIRONMENT VARIABLES

os.environ["ML_FLOW_TRACKING_URI"] = "http://localhost:8080"
os.environ["ML_FLOW_EXPERIMENT_NAME"] = f"xgboost_compressed_with_features_{time_stamp}"

# Folders
OUTPUT_DIR = Path("./TESTxgboost_results") / os.environ["ML_FLOW_EXPERIMENT_NAME"] 
os.environ["OUTPUT_DIR"] = str(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_DATA_DIR = Path("E:/data")  
os.environ["RAW_DATA_DIR"] = str(RAW_DATA_DIR)

# Models directory
os.environ["MODELS_DIR"] = str(OUTPUT_DIR / "models")
MODELS_DIR = Path(os.environ["MODELS_DIR"])
MODELS_DIR.mkdir(parents=True, exist_ok=True)

os.environ["FEATURES_DIR"] = str(OUTPUT_DIR / "features")
FEATURES_DIR = Path(os.environ["FEATURES_DIR"])
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

#Switches
EXTRACT_FEATURES = True  # Set to True to extract features, False to load existing features


In [ ]:
# Set MlFlow tracking URI
mlflow.set_tracking_uri(os.environ["ML_FLOW_TRACKING_URI"]) # Local: # mlflow server --host 127.0.0.1 --port 8080

In [ ]:
class DDMFeatureExtractor:
    def __init__(self):
        pass
    @staticmethod
    def gini(array):
            """Gini coefficient calculation"""
            array = np.sort(array)
            index = np.arange(1, array.shape[0] + 1)
            return (np.sum((2 * index - array.shape[0] - 1) * array)) / (array.shape[0] * np.sum(array))  
      
    def extract_ddm_features(self, fit_data: np.ndarray) -> pd.DataFrame:
        """
        Extract features from DDM data.
        """
        features = []

        for row in tqdm(fit_data, desc="Extracting DDM features"):
            f = {}
            x = np.array(row, dtype=np.float64) + 1e-10  # evita log(0)

            # 1. General statistics
            f['mean'] = np.mean(x)
            f['std'] = np.std(x)
            f['min'] = np.min(x)
            f['max'] = np.max(x)
            f['median'] = np.median(x)
            f['range'] = np.max(x) - np.min(x)
            f['skew'] = skew(x)
            f['kurtosis'] = kurtosis(x)
            f['entropy'] = entropy(x)
            f['gini'] = self.gini(x)

            # 2. Positional 
            f['peak_index'] = np.argmax(x)
            f['peak_value'] = np.max(x)
            f['center_of_mass'] = np.sum(np.arange(len(x)) * x) / np.sum(x)
            f['inertia'] = np.sum(((np.arange(len(x)) - f['center_of_mass'])**2) * x)

            # 3. Segmentations in thirds
            thirds = np.array_split(x, 3)
            for i, part in enumerate(thirds):
                f[f'sum_third_{i+1}'] = np.sum(part)
                f[f'mean_third_{i+1}'] = np.mean(part)
                f[f'max_third_{i+1}'] = np.max(part)

            # 3.1 Segmentations in windows of 5
            windows = np.array_split(x, 5)
            for i, w in enumerate(windows):
                f[f'mean_w{i+1}'] = np.mean(w)
                f[f'std_w{i+1}'] = np.std(w)
                f[f'max_w{i+1}'] = np.max(w)

            # 4. Derivative statistics and differences
            dx = np.diff(x)
            f['mean_diff'] = np.mean(dx)
            f['std_diff'] = np.std(dx)
            f['max_diff'] = np.max(dx)
            f['min_diff'] = np.min(dx)
            f['n_positive_diff'] = np.sum(dx > 0)
            f['n_negative_diff'] = np.sum(dx < 0)
            f['n_zero_diff'] = np.sum(dx == 0)

            # 5. Autocorrelations (lag 1-3)
            for lag in range(1, 4):
                ac = np.corrcoef(x[:-lag], x[lag:])[0, 1] if len(x) > lag else np.nan
                f[f'autocorr_lag{lag}'] = ac

            # 6. FFT 
            spectrum = np.abs(fft(x)) # type: ignore
            half_spectrum = spectrum[:len(spectrum)//2]  
            f['fft_peak_freq'] = np.argmax(half_spectrum)
            f['fft_max'] = np.max(half_spectrum)
            f['fft_median'] = np.median(half_spectrum)
            f['fft_mean'] = np.mean(half_spectrum)


            features.append(f)
        return features # type: ignore

In [ ]:
def combined_features_to_dataframe(combined_features, full_data = pd.DataFrame(), full_labels = pd.DataFrame()) -> Tuple[pd.DataFrame, pd.DataFrame, List[str]]:
    flat_features = [row[0] if isinstance(row, list) and len(row) > 0 else row for row in combined_features]
    FEATURES=list(combined_features[0][0].keys())
    combined_features = np.array([[row[key] for key in FEATURES] for row in flat_features])
    del flat_features
    combined_features.shape

    # Check for NaN and infinite values
    mask_finite = np.isfinite(combined_features).all(axis=1) & (np.abs(combined_features) < np.finfo(np.float64).max).all(axis=1)

    fit_data_with_features_clean = combined_features[mask_finite]
    labels_clean = full_labels[mask_finite]
    del combined_features
    return pd.DataFrame(fit_data_with_features_clean, columns=FEATURES),  pd.DataFrame(labels_clean, columns=['0']), FEATURES


In [ ]:
@dataclass
class SampleConfig:
    """Sample configuration."""
    name: str
    n_samples_per_class: int
    random_state: int = 42

@dataclass
class DatasetInfo:
    n_samples: int
    n_features: int
    class_distribution: Dict[int, int]
    features_columns: List[str]
    labels_columns: List[str]

@dataclass
class DataLoader:
    features_path: Union[str, Path]
    labels_path: Union[str, Path]
    _features_df_cache: Optional[pd.DataFrame] = field(default=None, init=False, repr=False)
    _labels_df_cache: Optional[pd.DataFrame] = field(default=None, init=False, repr=False)

    def __post_init__(self):
        self.features_path = Path(self.features_path)
        self.labels_path = Path(self.labels_path)

        if not self.features_path.exists():
            raise FileNotFoundError(f"Features file not found: {self.features_path}")
        if not self.labels_path.exists():
            raise FileNotFoundError(f"Labels file not found: {self.labels_path}")

    def load_full_data(self, use_cache: bool = False) -> Tuple[pd.DataFrame, pd.DataFrame]:
        
        if use_cache and self._features_df_cache is not None and self._labels_df_cache is not None:
            return self._features_df_cache, self._labels_df_cache

        features_df = pd.read_parquet(self.features_path)
        labels_df = pd.read_parquet(self.labels_path)

        # Dimensions check
        if len(features_df) != len(labels_df):
            raise ValueError(f"Features and labels have different lengths: "
                           f"{len(features_df)} vs {len(labels_df)}")

        if use_cache:
            self._features_df_cache = features_df
            self._labels_df_cache = labels_df

        return features_df, labels_df

    def clear_cache(self) -> None:
        self._features_df_cache = None
        self._labels_df_cache = None

    def load_balanced_sample(self, 
                            n_samples_per_class: int, 
                            random_state: int = 42,
                            use_cache: bool = False) -> Tuple[pd.DataFrame, pd.DataFrame]:
        
        # Load full data (with caching option)
        features_df_full, labels_df_full = self.load_full_data(use_cache=use_cache)

        # Get sampled indices balanced across classes
        sampled_indices = self._get_balanced_indices(
            labels_df_full = labels_df_full,
            n_samples_per_class = n_samples_per_class,
            random_state =random_state
        )
        features_df = features_df_full.loc[sampled_indices].reset_index(drop=True)
        labels_df = labels_df_full.loc[sampled_indices].reset_index(drop=True)

        return features_df, labels_df

    def load_balanced_sample_memory_efficient(self, n_samples_per_class: int, 
                                            random_state: int = 42) -> Tuple[pd.DataFrame, pd.DataFrame]:
        
        features_df_full = pd.read_parquet(self.features_path)
        labels_df_full = pd.read_parquet(self.labels_path)

        # Get sampled indices balanced across classes
        sampled_indices = self._get_balanced_indices(
            labels_df = labels_df_full, 
            n_samples_per_class = n_samples_per_class, 
            random_state = random_state
        )

        features_df = features_df_full.loc[sampled_indices].reset_index(drop=True)
        labels_df = labels_df_full.loc[sampled_indices].reset_index(drop=True)

        # Clean up memory
        del features_df_full, labels_df_full, sampled_indices

        return features_df, labels_df

    def load_max_balanced_sample(self, 
                                random_state: int = 42,
                                use_cache: bool = False) -> Tuple[pd.DataFrame, pd.DataFrame]:
        """
        Load the maximum number of samples with balanced labels.
        
        This method determines the size of the smallest class and samples 
        that many samples from each class, ensuring perfect balance.
        
        Args:
            random_state: Random seed for reproducible sampling
            use_cache: Whether to use cached data if available
            
        Returns:
            Tuple of (features_df, labels_df) with balanced sampling
            
        Raises:
            ValueError: If any class has 0 samples
        """
        # Load full data (with caching option)
        features_df_full, labels_df_full = self.load_full_data(use_cache=use_cache)
        
        # Get class distribution
        label_column = labels_df_full.iloc[:, 0]
        class_counts = label_column.value_counts()
        
        # Find the minimum class size
        min_class_size = class_counts.min()
        
        if min_class_size == 0:
            raise ValueError("One or more classes have 0 samples")
        
        # Get sampled indices using the minimum class size
        sampled_indices = self._get_balanced_indices(
            labels_df_full=labels_df_full,
            n_samples_per_class=min_class_size,
            random_state=random_state
        )
        
        features_df = features_df_full.loc[sampled_indices].reset_index(drop=True)
        labels_df = labels_df_full.loc[sampled_indices].reset_index(drop=True)
        
        return features_df, labels_df

    def load_max_balanced_sample_memory_efficient(self, 
                                                 random_state: int = 42) -> Tuple[pd.DataFrame, pd.DataFrame]:
        """
        Memory-efficient version of load_max_balanced_sample.
        
        This method loads data, determines the maximum balanced sample size,
        and immediately cleans up intermediate data structures to minimize memory usage.
        
        Args:
            random_state: Random seed for reproducible sampling
            
        Returns:
            Tuple of (features_df, labels_df) with balanced sampling
            
        Raises:
            ValueError: If any class has 0 samples
        """
        # Load data
        features_df_full = pd.read_parquet(self.features_path)
        labels_df_full = pd.read_parquet(self.labels_path)
        
        # Get class distribution
        label_column = labels_df_full.iloc[:, 0]
        class_counts = label_column.value_counts()
        
        # Find the minimum class size
        min_class_size = class_counts.min()
        
        if min_class_size == 0:
            raise ValueError("One or more classes have 0 samples")
        
        # Get sampled indices using the minimum class size
        sampled_indices = self._get_balanced_indices(
            labels_df=labels_df_full,
            n_samples_per_class=min_class_size,
            random_state=random_state
        )
        
        features_df = features_df_full.loc[sampled_indices].reset_index(drop=True)
        labels_df = labels_df_full.loc[sampled_indices].reset_index(drop=True)
        
        # Clean up memory
        del features_df_full, labels_df_full, sampled_indices, class_counts, label_column
        
        return features_df, labels_df

    def get_max_balanced_sample_info(self, use_cache: bool = False) -> Dict[str, Any]:
        """
        Get information about the maximum balanced sample without loading the actual data.
        
        Args:
            use_cache: Whether to use cached data if available
            
        Returns:
            Dictionary containing:
            - max_samples_per_class: Maximum samples that can be taken per class
            - total_balanced_samples: Total samples in the balanced dataset
            - class_distribution: Original class distribution
            - balanced_distribution: What the balanced distribution would be
        """
        # Load full data (with caching option)
        _, labels_df_full = self.load_full_data(use_cache=use_cache)
        
        # Get class distribution
        label_column = labels_df_full.iloc[:, 0]
        class_counts = label_column.value_counts().sort_index()
        
        # Find the minimum class size
        min_class_size = class_counts.min()
        
        # Create balanced distribution
        balanced_distribution = {class_label: min_class_size for class_label in class_counts.index}
        
        return {
            'max_samples_per_class': min_class_size,
            'total_balanced_samples': min_class_size * len(class_counts),
            'class_distribution': class_counts.to_dict(),
            'balanced_distribution': balanced_distribution,
            'n_classes': len(class_counts)
        }
    
    def _get_balanced_indices(self, labels_df: pd.DataFrame, 
                            n_samples_per_class: int, 
                            random_state: int = 42) -> pd.Index:
        """
        Get indices of a balanced sample from the labels DataFrame.
        """
        label_column = labels_df.iloc[:, 0]

        # Check if all classes have enough samples
        class_counts = label_column.value_counts()
        for class_label, count in class_counts.items():
            if count < n_samples_per_class:
                raise ValueError(f"Class {class_label} has only {count} samples, "
                               f"but {n_samples_per_class} requested")

        sampled_indices = (
            labels_df.groupby(label_column)
            .apply(lambda x: x.sample(n=n_samples_per_class, random_state=random_state))
            .index.get_level_values(1)
        )
        return sampled_indices

In [ ]:
class XGBoostPipeline:
    def __init__(self, features_df, labels_df, test_size=0.2, random_state=42, 
                 experiment_name="xgboost_binary_classification_noq", model_save_dir="./models_noq", save_voter=True):
        
        self.features_df = features_df
        self.labels_df = labels_df.values.ravel() if isinstance(labels_df, pd.DataFrame) else labels_df
        self.test_size = test_size
        self.random_state = random_state
        self.model = None
        self.best_params = None
        self.cv_results = None
        self.scaler = None
        self.save_voter = save_voter

        # MLflow setup
        self.experiment_name = experiment_name
        self.model_save_dir = model_save_dir
        self.run_id = None

        # Create output directory if it doesn't exist
        if not os.path.exists(model_save_dir):
            print(f"Creating model save directory: {model_save_dir}")
            os.makedirs(model_save_dir, exist_ok=True)

        # Setup MLflow experiment
        self._setup_mlflow_experiment()

    def _setup_mlflow_experiment(self):
        try:
            experiment = mlflow.get_experiment_by_name(self.experiment_name)
            if experiment is None:
                mlflow.create_experiment(self.experiment_name)

            mlflow.set_experiment(self.experiment_name)
            print(f"MLflow experiment '{self.experiment_name}' correctly set up.")

        except Exception as e:
            print(f"Error in mlflow configuration: {e}")

    def _log_to_mlflow(self, key, value):
        try:
            if isinstance(value, (int, float)):
                mlflow.log_metric(key, value)
            else:
                mlflow.log_param(key, value)
        except Exception as e:
            print(f"Error logging MLflow {key}: {e}")

    def prepare_data(self):
        """Split data into train and test sets"""
        print("=== Data preparation ===")

        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
            self.features_df, self.labels_df, 
            test_size=self.test_size, 
            random_state=self.random_state,
            stratify=self.labels_df
        )

        # Features scaling
        self.scaler = StandardScaler()
        self.X_train_scaled = self.scaler.fit_transform(self.X_train)
        self.X_test_scaled = self.scaler.transform(self.X_test)

        try:
            mlflow.log_param("dataset_total_samples", len(self.features_df))
            mlflow.log_param("train_samples", self.X_train.shape[0])
            mlflow.log_param("test_samples", self.X_test.shape[0])
            mlflow.log_param("n_features", self.X_train.shape[1])
            mlflow.log_param("test_size", self.test_size)
            mlflow.log_param("random_state", self.random_state)

            class_distribution = pd.Series(self.y_train).value_counts(normalize=True)
            for class_label, proportion in class_distribution.items():
                mlflow.log_metric(f"class_{class_label}_proportion", proportion)

        except Exception as e:
            print(f"Error mlflow: {e}")

        print("Dataset dimensions:")
        print(f"Train set: {self.X_train.shape[0]} campioni")
        print(f"Test set: {self.X_test.shape[0]} campioni")
        print(f"Features: {self.X_train.shape[1]}")
        print("\nClass distribution:")
        print(pd.Series(self.y_train).value_counts(normalize=True))

    def hyperparameter_tuning(self, cv_folds=5, verbose=True, n_iter=250):
        print("\n=== HYPERPARAMETER TUNING ===")

        self.param_distributions = {
            'n_estimators': [500, 1000, 1500],
            'max_depth': [3, 5, 7, 9],
            'learning_rate': [0.01, 0.001, 0.03, 0.005],
            'subsample': uniform(0.4, 0.6),     
            'colsample_bytree': uniform(0.4, 0.6), 
            #'colsample_bylevel': uniform(0.4, 0.6),
            #'colsample_bynode': uniform(0.4, 0.6), 
            'gamma': loguniform(1e-8, 1),
            'reg_alpha': loguniform(1e-8, 10),   
            'reg_lambda': loguniform(1e-8, 10),
            'min_child_weight': randint(1, 10),
            #'scale_pos_weight': uniform(0.5, 5.0),
            #'booster': ['gbtree', 'dart'],
        }

        try:
            mlflow.log_param("cv_folds", cv_folds)
            mlflow.log_param("search_iterations", n_iter)
            mlflow.log_param("search_scoring", "roc_auc")

            # Log hyperparameter search space
            for param, distribution in self.param_distributions.items():
                if hasattr(distribution, 'args'):
                    mlflow.log_param(f"search_space_{param}", str(distribution.args))
                else:
                    mlflow.log_param(f"search_space_{param}", str(distribution))

        except Exception as e:
            print(f"{e}")

        n_cores = multiprocessing.cpu_count()
        self.n_jobs = np.floor(0.75 * n_cores).astype(int)
        print(f"Using {self.n_jobs} cores for hyperparameter tuning (75% of {n_cores} total cores).")

        # Base model
        base_model = xgb.XGBClassifier(
            objective='binary:logistic',
            eval_metric='logloss',
            tree_method='hist',
            use_label_encoder=False,
            random_state=self.random_state,
            n_jobs=self.n_jobs
        )

        # Stratified K-Fold
        stratified_kfold = StratifiedKFold(
            n_splits=cv_folds, 
            shuffle=True, 
            random_state=self.random_state
        )

        # RandomizedSearchCV
        self.grid_search = RandomizedSearchCV(
            estimator=base_model,
            param_distributions=self.param_distributions,
            n_iter=n_iter,
            cv=stratified_kfold,
            scoring='roc_auc',
            n_jobs=self.n_jobs,
            verbose=2,
            random_state=self.random_state,
            return_train_score=True
        )

        
        print("Searching for hyperparameters!")
        self.grid_search.fit(self.X_train_scaled, self.y_train)

        self.best_params = self.grid_search.best_params_
        self.cv_results = pd.DataFrame(self.grid_search.cv_results_)

        try:
            mlflow.log_metric("best_cv_score", self.grid_search.best_score_)

            for param, value in self.best_params.items():
                mlflow.log_param(f"best_{param}", value)

            cv_results_path = os.path.join(self.model_save_dir, "cv_results.csv")
            self.cv_results.to_csv(cv_results_path, index=False)
            mlflow.log_artifact(cv_results_path, "hyperparameter_search")

        except Exception as e:
            print(f"Error in mlflow logging: {e}")

        print("\nBest hyperparameters found:")
        for param, value in self.best_params.items():
            print(f"  {param}: {value}")
        print(f"\nBest  ROC-AUC (CV): {self.grid_search.best_score_:.4f}")

        self._plot_hyperparameter_results()

    def train_final_model(self):
        """Finalize the model training with the best hyperparameters found"""
        print("\n=== Finalizing model ===")

        self.model = xgb.XGBClassifier(
            **self.best_params,
            objective='binary:logistic',
            eval_metric='logloss',
            random_state=self.random_state,
            n_jobs=self.n_jobs
        )

        eval_set = [(self.X_train_scaled, self.y_train), (self.X_test_scaled, self.y_test)]
        self.model.fit(
            self.X_train_scaled, self.y_train,
            eval_set=eval_set,
            verbose=False
        )

        self.y_pred = self.model.predict(self.X_test_scaled)
        self.y_pred_proba = self.model.predict_proba(self.X_test_scaled)[:, 1]

        metrics = {
            'accuracy': accuracy_score(self.y_test, self.y_pred),
            'precision': precision_score(self.y_test, self.y_pred),
            'recall': recall_score(self.y_test, self.y_pred),
            'f1_score': f1_score(self.y_test, self.y_pred),
            'roc_auc': auc(*roc_curve(self.y_test, self.y_pred_proba)[:2]),
            'average_precision': average_precision_score(self.y_test, self.y_pred_proba)
        }

        # Log metrics MLflow
        try:
            for metric_name, metric_value in metrics.items():
                mlflow.log_metric(f"test_{metric_name}", metric_value)

            
            mlflow.log_metric("n_estimators_used", self.model.n_estimators)

        except Exception as e:
            print(f"Error logging metrics: {e}")

        print("\nMetrics on test set:")
        for metric_name, metric_value in metrics.items():
            print(f"{metric_name.replace('_', ' ').title()}: {metric_value:.4f}")

    
    def save_model_and_scaler(self, model_name=None):
        print("\n=== Saving finalized model and scaler ===")

        if model_name is None:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            model_name = f"xgboost_model_{timestamp}"

        model_path = os.path.join(self.model_save_dir, f"{model_name}.joblib")
        scaler_path = os.path.join(self.model_save_dir, f"{model_name}_scaler.joblib")
        metadata_path = os.path.join(self.model_save_dir, f"{model_name}_metadata.json")

        try:
            # Save model and scaler
            joblib.dump(self.model, model_path)
            print(f"Model saved in: {model_path}")
            joblib.dump(self.scaler, scaler_path)
            print(f"Scaler saved in: {scaler_path}")
                
            # Save model and scaler to voters directory
            if self.save_voter:
                print("Saving voters...")
                joblib.dump(self.model, 'models/voters/xgboost_voter.joblib')
                joblib.dump(self.scaler, 'models/voters/xgboost_scaler.joblib')

            metadata = {
                'model_name': model_name,
                'timestamp': datetime.now().isoformat(),
                'best_params': self.best_params,
                'cv_score': float(self.grid_search.best_score_),
                'test_metrics': {
                    'accuracy': float(accuracy_score(self.y_test, self.y_pred)),
                    'precision': float(precision_score(self.y_test, self.y_pred)),
                    'recall': float(recall_score(self.y_test, self.y_pred)),
                    'f1_score': float(f1_score(self.y_test, self.y_pred)),
                    'roc_auc': float(auc(*roc_curve(self.y_test, self.y_pred_proba)[:2]))
                },
                'feature_names': list(self.features_df.columns),
                'n_features': len(self.features_df.columns),
                'train_samples': len(self.y_train),
                'test_samples': len(self.y_test)
            }

            with open(metadata_path, 'w') as f:
                json.dump(metadata, f, indent=2)
            print(f" Metadata saved: {metadata_path}")

            
            try:
                
                signature = infer_signature(self.X_train_scaled, self.y_pred_proba)
                mlflow.xgboost.log_model(
                    self.model, 
                    "xgboost_model",
                    signature=signature
                )
                
                mlflow.log_artifact(scaler_path, "preprocessing")
                mlflow.log_artifact(metadata_path, "model_info")

                mlflow.log_param("model_save_path", model_path)
                mlflow.log_param("scaler_save_path", scaler_path)
                mlflow.log_param("model_name", model_name)

                print("Model and scaler logged to MLflow successfully.")

            except Exception as e:
                print(f"Error logging MLflow: {e}")

            return model_path, scaler_path, metadata_path

        except Exception as e:
            print(f"Error saving: {e}")
            return None, None, None

    @staticmethod
    def load_model_and_scaler(model_path, scaler_path):
        try:
            model = joblib.load(model_path)
            scaler = joblib.load(scaler_path)
            print(f"Model loaded from: {model_path}")
            print(f"Scaler loaded from: {scaler_path}")
            return model, scaler
        except Exception as e:
            print(f"Error loading scaler or model: {e}")
            return None, None

    def predict_new_data(self, new_data, model_path=None, scaler_path=None):

        if model_path and scaler_path:
            model, scaler = self.load_model_and_scaler(model_path, scaler_path)
        else:
            model, scaler = self.model, self.scaler

        if model is None or scaler is None:
            print("Model or scaler not loaded. Cannot predict.")
            return None

        try:
            new_data_scaled = scaler.transform(new_data)

            predictions = model.predict(new_data_scaled)
            probabilities = model.predict_proba(new_data_scaled)[:, 1]

            return predictions, probabilities

        except Exception as e:
            print(f"Error during prediction: {e}")
            return None

    def finalize_model(self, features_df_to_finalize, labels_df_to_finalize, 
                      model_path, scaler_path, 
                      refit_scaler=bool,
                      final_model_name=None, experiment_suffix="_finalized"):
        """
        Loads a preexisting model and retrains it on the final data.
        """

        print("=== Finalizing model ===")

        labels_to_finalize = labels_df_to_finalize.values.ravel() if isinstance(labels_df_to_finalize, pd.DataFrame) else labels_df_to_finalize

        print(f"Loading model from: {model_path}")
        print(f"Loading scaler from: {scaler_path}")

        base_model, base_scaler = self.load_model_and_scaler(model_path, scaler_path)
        if base_model is None or base_scaler is None:
            print("Error in loading model or scaler")
            return None, None, None

        original_experiment_name = self.experiment_name
        finalization_experiment_name = original_experiment_name + experiment_suffix

        try:
            experiment = mlflow.get_experiment_by_name(finalization_experiment_name)
            if experiment is None:
                mlflow.create_experiment(finalization_experiment_name)
            mlflow.set_experiment(finalization_experiment_name)
        except Exception as e:
            print(f"Error setup MLflow: {e}")

        with mlflow.start_run() as run:
            finalization_run_id = run.info.run_id
            print(f"MLflow Finalization Run ID: {finalization_run_id}")

            try:
                mlflow.log_param("base_model_path", model_path)
                mlflow.log_param("base_scaler_path", scaler_path)
                mlflow.log_param("finalization_samples", len(features_df_to_finalize))
                mlflow.log_param("finalization_features", len(features_df_to_finalize.columns))
                mlflow.log_param("refit_scaler", refit_scaler)

                # Log distribuzione classi nei dati finali
                class_distribution = pd.Series(labels_to_finalize).value_counts(normalize=True)
                for class_label, proportion in class_distribution.items():
                    mlflow.log_metric(f"final_class_{class_label}_proportion", proportion)

                print(f"Final data: {len(features_df_to_finalize)} samples, {len(features_df_to_finalize.columns)} features")
                print("Class distribution in final data:")
                print(pd.Series(labels_to_finalize).value_counts(normalize=True))

                # Gestione scaler
                if refit_scaler:
                    print("Re-fitting scaler on new data...")
                    final_scaler = StandardScaler()
                    features_scaled = final_scaler.fit_transform(features_df_to_finalize)
                    mlflow.log_param("scaler_action", "refit_on_new_data")
                else:
                    print("Using existing scaler...")
                    final_scaler = base_scaler
                    features_scaled = final_scaler.transform(features_df_to_finalize)
                    mlflow.log_param("scaler_action", "use_existing")

                
                base_params = base_model.get_params()
                print(f"Base model parameters: {len(base_params)}")

                for param, value in base_params.items():
                    if param not in ['random_state', 'n_jobs', 'objective', 'eval_metric']:
                        mlflow.log_param(f"base_{param}", value)

                final_model = xgb.XGBClassifier(**base_params)

                # Training final model
                print("Starting final model training...")

                if len(features_df_to_finalize) > 100:
                    val_size = min(0.1, 100 / len(features_df_to_finalize))
                    X_train_final, X_val_final, y_train_final, y_val_final = train_test_split(
                        features_scaled, labels_to_finalize,
                        test_size=val_size,
                        random_state=self.random_state,
                        stratify=labels_to_finalize
                    )

                    eval_set = [(X_train_final, y_train_final), (X_val_final, y_val_final)]
                    final_model.fit(
                        X_train_final, y_train_final,
                        eval_set=eval_set,
                        verbose=False
                    )

                    mlflow.log_param("validation_split_used", True)
                    mlflow.log_param("validation_size", val_size)

                    y_val_pred = final_model.predict(X_val_final)
                    y_val_pred_proba = final_model.predict_proba(X_val_final)[:, 1]

                    final_metrics = {
                        'final_accuracy': accuracy_score(y_val_final, y_val_pred),
                        'final_precision': precision_score(y_val_final, y_val_pred),
                        'final_recall': recall_score(y_val_final, y_val_pred),
                        'final_f1_score': f1_score(y_val_final, y_val_pred),
                        'final_roc_auc': auc(*roc_curve(y_val_final, y_val_pred_proba)[:2])
                    }

                else:
                    final_model.fit(features_scaled, labels_to_finalize, verbose=False)
                    mlflow.log_param("validation_split_used", False)

                    y_train_pred = final_model.predict(features_scaled)
                    y_train_pred_proba = final_model.predict_proba(features_scaled)[:, 1]

                    final_metrics = {
                        'final_train_accuracy': accuracy_score(labels_to_finalize, y_train_pred),
                        'final_train_precision': precision_score(labels_to_finalize, y_train_pred),
                        'final_train_recall': recall_score(labels_to_finalize, y_train_pred),
                        'final_train_f1_score': f1_score(labels_to_finalize, y_train_pred),
                        'final_train_roc_auc': auc(*roc_curve(labels_to_finalize, y_train_pred_proba)[:2])
                    }

                
                for metric_name, metric_value in final_metrics.items():
                    mlflow.log_metric(metric_name, metric_value)

                print("\n Finalized model metrics:")
                for metric_name, metric_value in final_metrics.items():
                    print(f"  {metric_name.replace('_', ' ').title()}: {metric_value:.4f}")

                # Feature importance analysis
                print("\n Feature importance study...")
                base_importance = pd.DataFrame({
                    'feature': features_df_to_finalize.columns,
                    'base_importance': base_model.feature_importances_
                }).sort_values('base_importance', ascending=False)

                final_importance = pd.DataFrame({
                    'feature': features_df_to_finalize.columns,
                    'final_importance': final_model.feature_importances_
                }).sort_values('final_importance', ascending=False)

                importance_comparison = base_importance.merge(
                    final_importance, on='feature', how='inner'
                )
                importance_comparison['importance_change'] = (
                    importance_comparison['final_importance'] - importance_comparison['base_importance']
                )

                top_changes = importance_comparison.reindex(
                    importance_comparison['importance_change'].abs().sort_values(ascending=False).index
                ).head(10)

                print("Top 10 feature with greater change in importance between partial and full training data:")
                try:
                    for _, row in top_changes.iterrows():
                        change_pct = (row['importance_change'] / (row['base_importance'] + 1e-8)) * 100
                        print(f"  {row['feature']}: {row['importance_change']:+.4f} ({change_pct:+.1f}%)")
                except Exception as e:
                    print(f"Error printing feature importance changes: {e}")

                # Save model, scaler, and analysis results
                if final_model_name is None:
                    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
                    final_model_name = f"finalized_model_{timestamp}"
                    print(f"Final model name: {final_model_name}")

                final_model_path = os.path.join(self.model_save_dir, f"{final_model_name}.joblib")
                print(f"Final model will be saved as: {final_model_path}")
                final_scaler_path = os.path.join(self.model_save_dir, f"{final_model_name}_scaler.joblib")
                print(f"Final scaler will be saved as: {final_scaler_path}")
                final_metadata_path = os.path.join(self.model_save_dir, f"{final_model_name}_metadata.json")
                print(f"Final metadata will be saved as: {final_metadata_path}")
                importance_comparison_path = os.path.join(self.model_save_dir, f"{final_model_name}_importance_comparison.csv")
                print(f"Importance comparison will be saved as: {importance_comparison_path}")

                joblib.dump(final_model, final_model_path)
                joblib.dump(final_scaler, final_scaler_path)
                importance_comparison.to_csv(importance_comparison_path, index=False)

                # Final metadata
                final_metadata = {
                    'model_name': final_model_name,
                    'finalization_timestamp': datetime.now().isoformat(),
                    'base_model_path': model_path,
                    'base_scaler_path': scaler_path,
                    'finalization_samples': len(features_df_to_finalize),
                    'finalization_features': len(features_df_to_finalize.columns),
                    'refit_scaler': refit_scaler,
                    'final_metrics': {k: float(v) for k, v in final_metrics.items()},
                    'model_params': {k: v for k, v in base_params.items() if k not in ['random_state', 'n_jobs']},
                    'feature_names': list(features_df_to_finalize.columns),
                    'class_distribution': {str(k): float(v) for k, v in class_distribution.items()}
                }

                with open(final_metadata_path, 'w') as f:
                    json.dump(final_metadata, f, indent=2)
                print(f"Final metadata saved: {final_metadata_path}")

                # MLflow logging
                try:
                    signature = infer_signature(features_scaled, final_model.predict_proba(features_scaled))
                    mlflow.xgboost.log_model(
                        final_model, 
                        "finalized_xgboost_model",
                        signature=signature
                    )

                    # Log artifacts
                    mlflow.log_artifact(final_scaler_path, "preprocessing")
                    mlflow.log_artifact(final_metadata_path, "model_info")
                    mlflow.log_artifact(importance_comparison_path, "analysis")

                    # Log final model and scaler paths
                    mlflow.log_param("final_model_path", final_model_path)
                    mlflow.log_param("final_scaler_path", final_scaler_path)
                    mlflow.log_param("final_model_name", final_model_name)

                    fig, ax = plt.subplots(figsize=(12, 8))
                    top_15_comparison = importance_comparison.head(15)

                    x = np.arange(len(top_15_comparison))
                    width = 0.35

                    ax.bar(x - width/2, top_15_comparison['base_importance'], width, 
                          label='Modello Base', alpha=0.8)
                    ax.bar(x + width/2, top_15_comparison['final_importance'], width, 
                          label='Modello Finalizzato', alpha=0.8)

                    ax.set_xlabel('Features')
                    ax.set_ylabel('Importance Score')
                    ax.set_title('Confronto Feature Importance: Base vs Finalizzato')
                    ax.set_xticks(x)
                    ax.set_xticklabels(top_15_comparison['feature'], rotation=45, ha='right')
                    ax.legend()
                    ax.grid(True, alpha=0.3)
                    plt.tight_layout()
                    plots_dir = os.path.join(self.model_save_dir, "finalization_plots")
                    os.makedirs(plots_dir, exist_ok=True)
                    comparison_plot_path = os.path.join(plots_dir, f"{final_model_name}_importance_comparison.png")
                    plt.savefig(comparison_plot_path, dpi=300, bbox_inches='tight')
                    mlflow.log_artifact(comparison_plot_path, "plots")
                    plt.show()
                except Exception as e:
                    print(f"Error logging to MLflow: {e}")

                # Tag of the run
                mlflow.set_tag("pipeline_status", "finalized")
                mlflow.set_tag("model_type", "xgboost_finalized")
                mlflow.set_tag("base_model_source", model_path)

                print("\n=== Finalization complete ===")

                self.model = final_model
                self.scaler = final_scaler

                return final_model, final_scaler, finalization_run_id

            except Exception as e:
                mlflow.set_tag("pipeline_status", "failed")
                mlflow.log_param("error_message", str(e))
                print(f"Errore in finalization: {e}")
                raise
        try:
            mlflow.set_experiment(original_experiment_name)
        except:
            pass

    
    def plot_all_visualizations(self):
        print("\n=== Data Viz ===")

        try:
            # Crea directory temporanea per i grafici
            plots_dir = os.path.join(self.model_save_dir, "plots")
            os.makedirs(plots_dir, exist_ok=True)

            # 1. Feature Importance
            self._plot_feature_importance(save_dir=plots_dir)

            # 2. ROC
            self._plot_roc_curve(save_dir=plots_dir)

            # 3. Precision-Recall
            self._plot_pr_curve(save_dir=plots_dir)

            # 4. Confusion Matrix
            self._plot_confusion_matrix(save_dir=plots_dir)

            # 5. Predicted proba Distribution
            self._plot_probability_distribution(save_dir=plots_dir)

            # 6. Learning Curves
            self._plot_learning_curves(save_dir=plots_dir)

            # 7. Calibration Plot
            self._plot_calibration_curve(save_dir=plots_dir)

            # 8. Classification Report Heatmap
            self._plot_classification_report(save_dir=plots_dir)

            try:
                for plot_file in os.listdir(plots_dir):
                    if plot_file.endswith('.png'):
                        mlflow.log_artifact(os.path.join(plots_dir, plot_file), "plots")
                print("Plots saved on MLflow")
            except Exception as e:
                print(f"Error saving plots on MlFlow: {e}")

        except Exception as e:
            print(f"Error generating plots {e}")

    def _plot_hyperparameter_results(self, save_dir=None):
        fig, axes = plt.subplots(2, 2, figsize=(15, 12))

        #top_results = self.cv_results.nlargest(20, 'mean_test_score')

        # 1. Grid Search Convergence
        ax = axes[0, 0]
        ax.plot(range(len(self.cv_results)), self.cv_results['mean_test_score'])
        ax.axhline(y=self.grid_search.best_score_, color='r', linestyle='--', label='Best Score')
        ax.set_xlabel('Iteration')
        ax.set_ylabel('ROC-AUC Score')
        ax.set_title('Convergenza della Grid Search')
        ax.legend()
        ax.grid(True, alpha=0.3)

        # 2. Relative importance of parameters
        ax = axes[0, 1]
        param_importance = {}
        for param in self.best_params.keys():
            col = f'param_{param}'
            if col in self.cv_results.columns: 
                if self.cv_results[col].dtype == 'object':
                    unique_vals = self.cv_results[col].nunique()
                    param_importance[param] = unique_vals / len(self.cv_results)
                else:
                    correlation = self.cv_results[[col, 'mean_test_score']].corr().iloc[0, 1]
                    if not np.isnan(correlation):
                        param_importance[param] = abs(correlation)

        if param_importance:
            pd.Series(param_importance).sort_values().plot(kind='barh', ax=ax)
            ax.set_title('Relative Importance ')
            ax.set_xlabel('Importance score')

        # 3. Best Score Distribution
        ax = axes[1, 0]
        ax.hist(self.cv_results['mean_test_score'], bins=30, alpha=0.7, edgecolor='black')
        ax.axvline(self.grid_search.best_score_, color='red', linestyle='--', 
                  label=f'Best: {self.grid_search.best_score_:.4f}')
        ax.set_xlabel('ROC-AUC Score')
        ax.set_ylabel('Frequency')
        ax.set_title('Scores CV distribution')
        ax.legend()
        ax.grid(True, alpha=0.3)

        # 4. Training Time vs Performance
        ax = axes[1, 1]
        if 'mean_fit_time' in self.cv_results.columns:
            scatter = ax.scatter(self.cv_results['mean_fit_time'], 
                               self.cv_results['mean_test_score'],
                               c=self.cv_results['mean_test_score'],
                               cmap='viridis', alpha=0.6)
            ax.set_xlabel('Training time (seconds)')
            ax.set_ylabel('ROC-AUC Score')
            ax.set_title('Trade-off Time vs Performance')
            plt.colorbar(scatter, ax=ax)

        plt.tight_layout()

        if save_dir:
            plt.savefig(os.path.join(save_dir, 'hyperparameter_results.png'), 
                       dpi=300, bbox_inches='tight')
        plt.show()

    def _plot_feature_importance(self, save_dir=None):
        
        fig, axes = plt.subplots(1, 3, figsize=(20, 6))

        # 1. XGBoost native importance
        ax = axes[0]
        plot_importance(self.model, max_num_features=20, ax=ax, importance_type='weight')
        ax.set_title('Feature Importance - Weight (Usage frequency)')

        # 2. Gain importance
        ax = axes[1]
        plot_importance(self.model, max_num_features=20, ax=ax, importance_type='gain')
        ax.set_title('Feature Importance - Gain (Average gain of splits)')

        # 3. Cover importance
        ax = axes[2]
        plot_importance(self.model, max_num_features=20, ax=ax, importance_type='cover')
        ax.set_title('Feature Importance - Cover (Relative quantity of observations)')

        plt.tight_layout()

        if save_dir:
            plt.savefig(os.path.join(save_dir, 'feature_importance_types.png'), 
                       dpi=300, bbox_inches='tight')
        plt.show()

        fig, ax = plt.subplots(figsize=(10, 8))
        feature_importance = pd.DataFrame({
            'feature': self.features_df.columns,
            'importance': self.model.feature_importances_
        }).sort_values('importance', ascending=False).head(15)

        sns.barplot(data=feature_importance, y='feature', x='importance', ax=ax)
        ax.set_title('Top 15 Features')
        ax.set_xlabel('Importance Score')
        plt.tight_layout()

        if save_dir:
            plt.savefig(os.path.join(save_dir, 'top_features_importance.png'), 
                       dpi=300, bbox_inches='tight')
        plt.show()

    def _plot_roc_curve(self, save_dir=None):
        
        fig, ax = plt.subplots(figsize=(10, 8))

        fpr, tpr, thresholds = roc_curve(self.y_test, self.y_pred_proba)
        roc_auc = auc(fpr, tpr)

        # Plot
        ax.plot(fpr, tpr, color='darkorange', lw=2, 
                label=f'ROC curve (AUC = {roc_auc:.3f})')
        ax.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random')

        # Find optimal threshold
        optimal_idx = np.argmax(tpr - fpr)
        optimal_threshold = thresholds[optimal_idx]
        ax.scatter(fpr[optimal_idx], tpr[optimal_idx], color='red', s=100, 
                  label=f'Optimal point (threshold={optimal_threshold:.3f})')

        ax.set_xlim([0.0, 1.0])
        ax.set_ylim([0.0, 1.05])
        ax.set_xlabel('False Positive Rate')
        ax.set_ylabel('True Positive Rate')
        ax.set_title('Receiver Operating Characteristic (ROC) Curve')
        ax.legend(loc="lower right")
        ax.grid(True, alpha=0.3)

        plt.tight_layout()

        if save_dir:
            plt.savefig(os.path.join(save_dir, 'roc_curve.png'), 
                       dpi=300, bbox_inches='tight')
        plt.show()

    def _plot_pr_curve(self, save_dir=None):
        fig, ax = plt.subplots(figsize=(10, 8))

        precision, recall, thresholds = precision_recall_curve(self.y_test, self.y_pred_proba)
        average_precision = average_precision_score(self.y_test, self.y_pred_proba)

        # Plot PR Curve
        ax.plot(recall, precision, color='blue', lw=2,
                label=f'PR curve (AP = {average_precision:.3f})')

        # Baseline 
        baseline = np.sum(self.y_test) / len(self.y_test)
        ax.axhline(y=baseline, color='red', linestyle='--', 
                  label=f'Baseline (Random) = {baseline:.3f}')

        ax.set_xlim([0.0, 1.0])
        ax.set_ylim([0.0, 1.05])
        ax.set_xlabel('Recall')
        ax.set_ylabel('Precision')
        ax.set_title('Precision-Recall Curve')
        ax.legend(loc="lower left")
        ax.grid(True, alpha=0.3)

        plt.tight_layout()

        if save_dir:
            plt.savefig(os.path.join(save_dir, 'precision_recall_curve.png'), 
                       dpi=300, bbox_inches='tight')
        plt.show()

    def _plot_confusion_matrix(self, save_dir=None):
        """Visualizza la matrice di confusione"""
        fig, axes = plt.subplots(1, 2, figsize=(15, 6))

        cm = confusion_matrix(self.y_test, self.y_pred)

        # 1. Confusion matrix with absolute values
        ax = axes[0]
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax)
        ax.set_xlabel('Predicted')
        ax.set_ylabel('Actual')
        ax.set_title('Confusion Matrix - Valori Assoluti')

        # 2. Normalized confusion matrix
        ax = axes[1]
        cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        sns.heatmap(cm_normalized, annot=True, fmt='.2%', cmap='Greens', ax=ax)
        ax.set_xlabel('Predicted')
        ax.set_ylabel('Actual')
        ax.set_title('Confusion Matrix - Normalizzata per Riga')
        plt.tight_layout()
        if save_dir:
            plt.savefig(os.path.join(save_dir, 'confusion_matrix.png'), 
                       dpi=300, bbox_inches='tight')
        plt.show()

    def _plot_probability_distribution(self, save_dir=None):
        fig, axes = plt.subplots(1, 2, figsize=(15, 6))

        # 1. Class probability distribution
        ax = axes[0]
        ax.hist(self.y_pred_proba[self.y_test == 0], bins=50, alpha=0.5, 
                label='Classe 0', color='blue', density=True)
        ax.hist(self.y_pred_proba[self.y_test == 1], bins=50, alpha=0.5, 
                label='Classe 1', color='red', density=True)
        ax.axvline(x=0.5, color='black', linestyle='--', label='Threshold=0.5')
        ax.set_xlabel('Predicted Probability')
        ax.set_ylabel('Density')
        ax.set_title('Probability Distribution per Class')
        ax.legend()

        # 2. Probability box plot
        ax = axes[1]
        data_to_plot = [self.y_pred_proba[self.y_test == 0], 
                       self.y_pred_proba[self.y_test == 1]]
        ax.boxplot(data_to_plot, labels=['Classe 0', 'Classe 1'])
        ax.set_ylabel('Predicted Probability')
        ax.set_title('Probability Box Plot per Class')
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        if save_dir:
            plt.savefig(os.path.join(save_dir, 'probability_distribution.png'), 
                       dpi=300, bbox_inches='tight')
        plt.show()

    def _plot_learning_curves(self, save_dir=None):
        _, ax = plt.subplots(figsize=(10, 8))
        results = self.model.evals_result()

        if results:
            epochs = len(results['validation_0']['logloss'])
            x_axis = range(0, epochs)

            # Plot training & validation loss
            ax.plot(x_axis, results['validation_0']['logloss'], label='Train')
            ax.plot(x_axis, results['validation_1']['logloss'], label='Test')
            ax.set_xlabel('Epoca')
            ax.set_ylabel('Log Loss')
            ax.set_title('Learning Curves - Training Log Loss')
            ax.legend()
            ax.grid(True, alpha=0.3)

        plt.tight_layout()

        if save_dir:
            plt.savefig(os.path.join(save_dir, 'learning_curves.png'), 
                       dpi=300, bbox_inches='tight')
        plt.show()

    def _plot_calibration_curve(self, save_dir=None):
        from sklearn.calibration import calibration_curve
        _, ax = plt.subplots(figsize=(10, 8))

        fraction_of_positives, mean_predicted_value = calibration_curve(
            self.y_test, self.y_pred_proba, n_bins=10
        )

        # Plot
        ax.plot(mean_predicted_value, fraction_of_positives, 's-', label='XGBoost')
        ax.plot([0, 1], [0, 1], 'k--', label='Perfectly calibrated')
        ax.set_xlabel('Mean Predicted Probability')
        ax.set_ylabel('Fraction of Positives')
        ax.set_title('Calibration Plot (Reliability Diagram)')
        ax.legend()
        ax.grid(True, alpha=0.3)

        plt.tight_layout()

        if save_dir:
            plt.savefig(os.path.join(save_dir, 'calibration_curve.png'), 
                       dpi=300, bbox_inches='tight')
        plt.show()

    def _plot_classification_report(self, save_dir=None):
        _, ax = plt.subplots(figsize=(8, 6))

        report = classification_report(self.y_test, self.y_pred, output_dict=True)
        df_report = pd.DataFrame(report).transpose()
        df_report = df_report.iloc[:-1, :-1]

        # Heatmap
        sns.heatmap(df_report, annot=True, fmt='.3f', cmap='YlGnBu', ax=ax)
        ax.set_title('Classification Report Heatmap')

        plt.tight_layout()

        if save_dir:
            plt.savefig(os.path.join(save_dir, 'classification_report.png'), 
                       dpi=300, bbox_inches='tight')
        plt.show()

    def run_complete_pipeline(self, model_name=None):
        print("=== Starting complete pipeline ===\n")

        with mlflow.start_run() as run:
            self.run_id = run.info.run_id
            print(f"🚀 MLflow Run ID: {self.run_id}")

            try:
                self.prepare_data()
                self.hyperparameter_tuning()
                self.train_final_model()
                model_path, scaler_path, _ = self.save_model_and_scaler(model_name)
                self.plot_all_visualizations()
                
                mlflow.set_tag("pipeline_status", "completed")
                mlflow.set_tag("model_type", "xgboost_binary_classifier")

                print("\n=== PIPELINE COMPLETED ===")
                print(f"MLflow Run: {self.run_id}")
                print(f"Modello saved in: {model_path}")
                print(f"Scaler saved in: {scaler_path}")

                return self.model, self.best_params, self.run_id

            except Exception as e:
                mlflow.set_tag("pipeline_status", "failed")
                mlflow.log_param("error_message", str(e))
                print(f"Error pipeline: {e}")
                raise


In [ ]:
# Load data from Parquet
parquet_path = f"{RAW_DATA_DIR}/balanced_df_enh_encoder_10M.parquet"
df_parquet = pd.read_parquet(parquet_path)
fit_data_full = df_parquet.drop(columns=['label']).values
fit_labels_full = df_parquet['label'].values

#Load test data from Parquet
parquet_path_test = f"{RAW_DATA_DIR}/test_df_enh_encoder_2M.parquet"
df_parquet_test = pd.read_parquet(parquet_path_test)
test_data = df_parquet_test.drop(columns=['label']).values
test_labels = df_parquet_test['label'].values

In [ ]:
from collections import Counter
Counter(test_labels)

In [ ]:
cpu_cores = multiprocessing.cpu_count()
print(f"Available {cpu_cores} CPU cores for parallel processing.")

In [ ]:
if EXTRACT_FEATURES:
    # Feature extraction train set
    features_extractor = DDMFeatureExtractor()

    def extract_ddm_features_row(row):
        return features_extractor.extract_ddm_features(np.array([row]))

    combined_features_train = Parallel(n_jobs=cpu_cores-2, backend="loky")(delayed(extract_ddm_features_row)(row) for row in tqdm(fit_data_full, desc="Estrazione features train"))


    fit_data_with_features_df, labels_clean_df, FEATURES = combined_features_to_dataframe(combined_features_train, 
                                                                                            full_data=fit_data_full, 
                                                                                            full_labels=fit_labels_full)

    #fit_data_with_features_df.to_csv(f"{FEATURES_DIR}/features_extracted_enh_enc_5M.csv", index=False)
    fit_data_with_features_df.to_parquet(f"{FEATURES_DIR}/features_extracted_enh_enc_10M.parquet", index=False)
    labels_clean_df.rename(columns={'0': 'label'}, inplace=True)
    labels_clean_df.to_parquet(f"{FEATURES_DIR}/labels_extracted_enh_enc_10M.parquet", index=False)


    # Feature extraction test set
    features_extractor = DDMFeatureExtractor()

    def extract_ddm_features_row(row):
        return features_extractor.extract_ddm_features(np.array([row]))

    combined_features_test = Parallel(n_jobs=cpu_cores-2, backend="loky")(delayed(extract_ddm_features_row)(row) for row in tqdm(test_data, desc="Estrazione features test"))

    test_data_with_features_df, test_labels_clean_df, FEATURES = combined_features_to_dataframe(combined_features_test, 
                                                                                                full_data=test_data, 
                                                                                                full_labels=test_labels)

    test_data_with_features_df.to_parquet(f"{FEATURES_DIR}/test_features_extracted_enh_enc_2M.parquet", index=False)
    test_labels_clean_df.rename(columns={'0': 'label'}, inplace=True)
    test_labels_clean_df.to_parquet(f"{FEATURES_DIR}/test_labels_extracted_enh_enc_2M.parquet", index=False)

else:
    # Load extracted features from Parquet
    fit_data_with_features_df = pd.read_parquet(f"{FEATURES_DIR}/features_extracted_enh_enc_10M.parquet")
    fit_labels_clean_df = pd.read_parquet(f"{FEATURES_DIR}/labels_extracted_enh_enc_10M.parquet")
    fit_labels_clean = fit_labels_clean_df['label'].values

    test_data_with_features_df = pd.read_parquet(f"{FEATURES_DIR}/test_features_extracted_enh_enc_2M.parquet")
    test_labels_clean_df = pd.read_parquet(f"{FEATURES_DIR}/test_labels_extracted_enh_enc_2M.parquet")
    test_labels_clean = test_labels_clean_df['label'].values
    FEATURES = fit_data_with_features_df.columns.tolist()

In [ ]:
loader = DataLoader(
features_path=f"{FEATURES_DIR}/features_extracted_enh_enc_10M.parquet"
labels_path=f"{FEATURES_DIR}/labels_extracted_enh_enc_10M.parquet"
)

# Small balanced sample for hyperparameter tuning
features_df, labels_df = loader.load_balanced_sample_memory_efficient(n_samples_per_class=1000000, random_state=42)
print(f"Training set shape: features={features_df.shape}, labels={labels_df.shape}")

# Large balanced sample for final retraining
features_df_to_finalize, labels_df_to_finalize = loader.load_max_balanced_sample_memory_efficient(
    random_state=42
)
print(f"Final set shape: features={features_df_to_finalize.shape}, labels={labels_df_to_finalize.shape}")
print(f"Class distribution in dev set set: {labels_df.iloc[:, 0].value_counts()}")
print(f"Class distribution in final set: {labels_df_to_finalize.iloc[:, 0].value_counts()}")

In [ ]:
features_df

In [ ]:
from collections import Counter
Counter(test_labels_clean_df['label'])

In [ ]:
# Model trained on a smaller dataset for hyperparameter tuning and development
dev_model_name = f"{os.environ['ML_FLOW_EXPERIMENT_NAME']}_dev_run_{time_stamp}"
dev_model_path = f'{os.environ["MODELS_DIR"]}/{dev_model_name}.joblib'

# Final model trained on the largest balanced dataset
final_model_name = f"{os.environ['ML_FLOW_EXPERIMENT_NAME']}_final_run_{time_stamp}.joblib"
final_model_path = f'{os.environ["MODELS_DIR"]}/{final_model_name}.joblib'

scaler_path = f'{os.environ["MODELS_DIR"]}/{dev_model_name}_scaler.joblib'

In [ ]:
# Starting pipeline
pipeline = XGBoostPipeline(
    features_df=features_df,
    labels_df=labels_df,
    experiment_name=os.environ['ML_FLOW_EXPERIMENT_NAME'], 
    model_save_dir=dev_model_path
)

# Run pipeline
model, best_params, run_id = pipeline.run_complete_pipeline(
    model_name=dev_model_name
)

In [ ]:
# Finalizing model with full data
final_model, final_scaler, final_run_id = pipeline.finalize_model(
     features_df_to_finalize=fit_data_with_features_df ,
     labels_df_to_finalize=labels_clean_df,
     model_path=dev_model_path,
     scaler_path=scaler_path,
     refit_scaler=False,  # Retrain scaler on full data
     final_model_name=final_model_path,
 )

### DEEPTEST

In [ ]:
xg_boost_model_path = final_model_path
xg_boost_scaler_path = scaler_path

In [ ]:
import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings('ignore')

class DeepTest:
    """
    Classe avanzata per testare modelli XGBoost su múltipli test set
    con metriche complete e visualizzazioni professionali
    """
    
    def __init__(self, model_path, test_data, test_labels):
        """
        Inizializza DeepTest per XGBoost
        
        Parameters:
        -----------
        model_path : str
            Path del modello XGBoost salvato (.pkl, .json, .model)
        test_data : pd.DataFrame
            DataFrame con i dati di test
        test_labels : np.array
            Array con le label di test
        """
        self.model_path = model_path
        self.test_data = test_data
        self.test_labels = test_labels
        self.model = None
        self.results = {}
        self.detailed_results = []
        
        # Configurazione colori per grafici professionali
        self.colors = {
            'primary': '#2E86AB',
            'secondary': '#A23B72', 
            'accent': '#F18F01',
            'success': '#C73E1D',
            'neutral': '#6C757D',
            'light': '#F8F9FA'
        }
        
        # Stile professionale
        plt.style.use('default')
        sns.set_palette([self.colors['primary'], self.colors['secondary'], 
                        self.colors['accent'], self.colors['success']])
        
        print("🚀 DeepTest inizializzato per XGBoost")
        print(f"   📊 Dati test: {test_data.shape}")
        print(f"   🎯 Label: {len(test_labels)} ({np.sum(test_labels)} positivi)")
    
    def load_model(self, model_format='pickle'):
        """
        Carica il modello XGBoost
        
        Parameters:
        -----------
        model_format : str
            Formato del modello: 'pickle', 'json', 'xgb_native'
        """
        try:
            if model_format == 'pickle':
                with open(self.model_path, 'rb') as f:
                    self.model = pickle.load(f)
            elif model_format == 'json':
                self.model = xgb.XGBClassifier()
                self.model.load_model(self.model_path)
            elif model_format == 'xgb_native':
                self.model = xgb.Booster()
                self.model.load_model(self.model_path)
            else:
                raise ValueError(f"Formato non supportato: {model_format}")
            
            print(f"✅ Modello XGBoost caricato da: {self.model_path}")
            
            # Verifica se è un XGBClassifier o Booster
            if hasattr(self.model, 'predict_proba'):
                self.model_type = 'classifier'
                print("   📋 Tipo: XGBClassifier")
            else:
                self.model_type = 'booster'
                print("   📋 Tipo: Booster (predizioni raw)")
            
        except Exception as e:
            print(f"❌ Errore nel caricamento del modello: {e}")
            raise
    
    def create_test_splits(self, n_splits=10, random_state=42):
        """
        Crea N split stratificati dei dati di test
        
        Parameters:
        -----------
        n_splits : int
            Numero di split da creare
        random_state : int
            Seed per riproducibilità
            
        Returns:
        --------
        list : Lista di tuple (X_split, y_split)
        """
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
        splits = []
        
        for train_idx, test_idx in skf.split(self.test_data, self.test_labels):
            X_split = self.test_data.iloc[test_idx].copy()
            y_split = self.test_labels[test_idx].copy()
            splits.append((X_split, y_split))
        
        print(f"📋 Creati {n_splits} split stratificati")
        return splits
    
    def _predict_xgboost(self, X):
        """
        Gestisce le predizioni per diversi tipi di modelli XGBoost
        
        Returns:
        --------
        tuple : (y_pred, y_pred_proba)
        """
        if self.model_type == 'classifier':
            # XGBClassifier
            y_pred_proba = self.model.predict_proba(X)[:, 1]
            y_pred = self.model.predict(X)
        else:
            # Booster - serve DMatrix
            dtest = xgb.DMatrix(X)
            y_pred_proba = self.model.predict(dtest)
            y_pred = (y_pred_proba > 0.5).astype(int)
        
        return y_pred, y_pred_proba
    
    def calculate_metrics(self, y_true, y_pred, y_pred_proba):
        """
        Calcola tutte le metriche di classificazione
        
        Returns:
        --------
        dict : Dizionario con tutte le metriche
        """
        metrics = {}
        
        # Metriche base
        metrics['accuracy'] = accuracy_score(y_true, y_pred)
        metrics['precision'] = precision_score(y_true, y_pred, zero_division=0)
        metrics['recall'] = recall_score(y_true, y_pred, zero_division=0)
        metrics['f1'] = f1_score(y_true, y_pred, zero_division=0)
        metrics['specificity'] = precision_score(y_true, y_pred, pos_label=0, zero_division=0)
        
        # Metriche avanzate
        try:
            metrics['auc_roc'] = roc_auc_score(y_true, y_pred_proba)
            metrics['auc_pr'] = average_precision_score(y_true, y_pred_proba)
            
            # Clip probabilità per evitare log(0)
            y_pred_proba_clipped = np.clip(y_pred_proba, 1e-15, 1-1e-15)
            metrics['log_loss'] = log_loss(y_true, y_pred_proba_clipped)
            metrics['brier_score'] = brier_score_loss(y_true, y_pred_proba)
        except:
            metrics['auc_roc'] = 0.5
            metrics['auc_pr'] = np.mean(y_true)
            metrics['log_loss'] = np.inf
            metrics['brier_score'] = np.inf
        
        # Matthews Correlation Coefficient
        metrics['mcc'] = matthews_corrcoef(y_true, y_pred)
        
        # Matrice di confusione
        cm = confusion_matrix(y_true, y_pred)
        if cm.shape == (2, 2):
            tn, fp, fn, tp = cm.ravel()
            metrics['true_positives'] = tp
            metrics['true_negatives'] = tn
            metrics['false_positives'] = fp
            metrics['false_negatives'] = fn
            metrics['sensitivity'] = tp / (tp + fn) if (tp + fn) > 0 else 0
            metrics['specificity'] = tn / (tn + fp) if (tn + fp) > 0 else 0
        else:
            metrics.update({
                'true_positives': 0, 'true_negatives': 0,
                'false_positives': 0, 'false_negatives': 0,
                'sensitivity': 0, 'specificity': 0
            })
        
        return metrics
    
    def run_comprehensive_test(self, n_splits=10, verbose=True):
        """
        Esegue test completo su N split con tutte le metriche
        
        Parameters:
        -----------
        n_splits : int
            Numero di split per il test
        verbose : bool
            Mostra progress durante il test
            
        Returns:
        --------
        dict : Risultati aggregati del test
        """
        if self.model is None:
            raise ValueError("❌ Carica prima il modello con load_model()")
        
        print(f"\n🔬 Inizio test completo XGBoost su {n_splits} split...")
        
        # Crea gli split
        splits = self.create_test_splits(n_splits)
        
        # Lista per salvare tutti i risultati
        all_metrics = []
        all_predictions = []
        all_probabilities = []
        all_true_labels = []
        
        # Testa su ogni split
        for i, (X_split, y_split) in enumerate(splits):
            if verbose:
                print(f"   Split {i+1}/{n_splits}: {X_split.shape[0]} samples", end=" -> ")
            
            try:
                # Predizioni con gestione dei diversi tipi di modello XGBoost
                y_pred, y_pred_proba = self._predict_xgboost(X_split)
                
                # Calcola metriche
                metrics = self.calculate_metrics(y_split, y_pred, y_pred_proba)
                metrics['split_id'] = i
                metrics['n_samples'] = len(y_split)
                
                all_metrics.append(metrics)
                all_predictions.extend(y_pred)
                all_probabilities.extend(y_pred_proba)
                all_true_labels.extend(y_split)
                
                if verbose:
                    print(f"Acc: {metrics['accuracy']:.3f}, AUC: {metrics['auc_roc']:.3f}")
                
            except Exception as e:
                print(f"❌ Errore nello split {i+1}: {e}")
                continue
        
        # Converti in DataFrame per analisi più facile
        metrics_df = pd.DataFrame(all_metrics)
        
        # Calcola statistiche aggregate
        aggregate_stats = {}
        metric_names = [col for col in metrics_df.columns 
                       if col not in ['split_id', 'n_samples']]
        
        for metric in metric_names:
            aggregate_stats[f'{metric}_mean'] = metrics_df[metric].mean()
            aggregate_stats[f'{metric}_std'] = metrics_df[metric].std()
            aggregate_stats[f'{metric}_min'] = metrics_df[metric].min()
            aggregate_stats[f'{metric}_max'] = metrics_df[metric].max()
        
        # Salva risultati
        self.results = {
            'individual_results': metrics_df,
            'aggregate_stats': aggregate_stats,
            'all_predictions': np.array(all_predictions),
            'all_probabilities': np.array(all_probabilities),
            'all_true_labels': np.array(all_true_labels),
            'n_splits': n_splits
        }
        
        print("✅ Test XGBoost completato!")
        print(f"Accuracy media: {aggregate_stats['accuracy_mean']:.4f} ± {aggregate_stats['accuracy_std']:.4f}")
        print(f"AUC media: {aggregate_stats['auc_roc_mean']:.4f} ± {aggregate_stats['auc_roc_std']:.4f}")
        
        return self.results
    
    def get_feature_importance(self, importance_type='gain', max_features=20):
        """
        Ottiene e visualizza feature importance da XGBoost
        
        Parameters:
        -----------
        importance_type : str
            Tipo di importance: 'gain', 'weight', 'cover'
        max_features : int
            Numero massimo di features da mostrare
        """
        if self.model is None:
            print("❌ Carica prima il modello")
            return None
        
        try:
            if self.model_type == 'classifier':
                # XGBClassifier
                if hasattr(self.model, 'feature_importances_'):
                    importance = self.model.feature_importances_
                    feature_names = self.test_data.columns
                else:
                    importance = self.model.get_booster().get_score(importance_type=importance_type)
                    feature_names = list(importance.keys())
                    importance = list(importance.values())
            else:
                # Booster
                importance_dict = self.model.get_score(importance_type=importance_type)
                feature_names = list(importance_dict.keys())
                importance = list(importance_dict.values())
            
            # Crea DataFrame e ordina
            feature_imp_df = pd.DataFrame({
                'feature': feature_names,
                'importance': importance
            }).sort_values('importance', ascending=False).head(max_features)
            
            # Visualizza
            plt.figure(figsize=(12, 8))
            plt.barh(range(len(feature_imp_df)), feature_imp_df['importance'], 
                    color=self.colors['primary'], alpha=0.8)
            plt.yticks(range(len(feature_imp_df)), feature_imp_df['feature'])
            plt.xlabel(f'Feature Importance ({importance_type})')
            plt.title(f'Top {max_features} XGBoost Feature Importance', 
                     fontsize=14, fontweight='bold')
            plt.gca().invert_yaxis()
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()
            
            return feature_imp_df
            
        except Exception as e:
            print(f"❌ Errore nel calcolo feature importance: {e}")
            return None
    
    def print_detailed_report(self):
        """
        Stampa report dettagliato dei risultati XGBoost
        """
        if not self.results:
            print("❌ Esegui prima run_comprehensive_test()")
            return
        
        stats = self.results['aggregate_stats']
        
        print("\n" + "="*60)
        print("XGBOOST DEEP TEST REPORT - METRICHE AGGREGATE")
        print("="*60)
        
        # Metriche principali
        main_metrics = [
            ('Accuracy', 'accuracy'),
            ('Precision', 'precision'),
            ('Recall', 'recall'),
            ('F1-Score', 'f1'),
            ('AUC-ROC', 'auc_roc'),
            ('AUC-PR', 'auc_pr'),
            ('MCC', 'mcc')
        ]
        
        print("METRICHE PRINCIPALI:")
        for name, key in main_metrics:
            mean_val = stats[f'{key}_mean']
            std_val = stats[f'{key}_std']
            print(f"   {name:<12}: {mean_val:.4f} ± {std_val:.4f}")
        
        # Confusion Matrix aggregata
        print(f"CONFUSION MATRIX (Media su {self.results['n_splits']} split):")
        tp = stats['true_positives_mean']
        tn = stats['true_negatives_mean'] 
        fp = stats['false_positives_mean']
        fn = stats['false_negatives_mean']
        
        print(f"   True Positives : {tp:.1f} ± {stats['true_positives_std']:.1f}")
        print(f"   True Negatives : {tn:.1f} ± {stats['true_negatives_std']:.1f}")
        print(f"   False Positives: {fp:.1f} ± {stats['false_positives_std']:.1f}")
        print(f"   False Negatives: {fn:.1f} ± {stats['false_negatives_std']:.1f}")
        
        # Metriche di loss
        print(f"METRICHE DI LOSS:")
        print(f"   Log Loss    : {stats['log_loss_mean']:.4f} ± {stats['log_loss_std']:.4f}")
        print(f"   Brier Score : {stats['brier_score_mean']:.4f} ± {stats['brier_score_std']:.4f}")
    
    def create_professional_visualizations(self, figsize=(20, 15), save_path=None):
        """
        Crea visualizzazioni professionali per XGBoost
        
        Parameters:
        -----------
        figsize : tuple
            Dimensioni della figura
        save_path : str
            Path per salvare la figura (opzionale)
        """
        if not self.results:
            print("❌ Esegui prima run_comprehensive_test()")
            return
        
        # Configura la figura
        fig = plt.figure(figsize=figsize)
        fig.suptitle('XGBoost Binary Classifier - Comprehensive Performance Analysis', 
                     fontsize=20, fontweight='bold', y=0.98)
        
        # Layout: 3x3 grid
        gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3, 
                             left=0.06, right=0.94, top=0.92, bottom=0.06)
        
        # 1. Metriche principali - Boxplot
        ax1 = fig.add_subplot(gs[0, :2])
        self._plot_metrics_boxplot(ax1)
        
        # 2. Confusion Matrix media
        ax2 = fig.add_subplot(gs[0, 2])
        self._plot_avg_confusion_matrix(ax2)
        
        # 3. ROC Curve
        ax3 = fig.add_subplot(gs[1, 0])
        self._plot_roc_curve(ax3)
        
        # 4. Precision-Recall Curve  
        ax4 = fig.add_subplot(gs[1, 1])
        self._plot_precision_recall_curve(ax4)
        
        # 5. Distribuzione delle probabilità
        ax5 = fig.add_subplot(gs[1, 2])
        self._plot_probability_distribution(ax5)
        
        # 6. Stabilità delle metriche
        ax6 = fig.add_subplot(gs[2, 0])
        self._plot_metrics_stability(ax6)
        
        # 7. Accuracy vs AUC scatter
        ax7 = fig.add_subplot(gs[2, 1])
        self._plot_accuracy_vs_auc(ax7)
        
        # 8. Tabella riassuntiva
        ax8 = fig.add_subplot(gs[2, 2])
        self._plot_summary_table(ax8)
        
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight', 
                       facecolor='white', edgecolor='none')
            print(f"📊 Visualizzazioni salvate in: {save_path}")
        
        plt.tight_layout()
        plt.show()
    
    def _plot_metrics_boxplot(self, ax):
        """Boxplot delle metriche principali"""
        df = self.results['individual_results']
        metrics = ['accuracy', 'precision', 'recall', 'f1', 'auc_roc']
        
        data_to_plot = [df[metric].values for metric in metrics]
        labels = ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC-ROC']
        
        bp = ax.boxplot(data_to_plot, labels=labels, patch_artist=True)
        
        colors = [self.colors['primary'], self.colors['secondary'], 
                 self.colors['accent'], self.colors['success'], self.colors['neutral']]
        
        for patch, color in zip(bp['boxes'], colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)
        
        ax.set_title('Distribution of Key Metrics Across Test Splits', 
                    fontsize=14, fontweight='bold')
        ax.set_ylabel('Score')
        ax.grid(True, alpha=0.3)
        ax.set_ylim(0, 1.05)
    
    def _plot_avg_confusion_matrix(self, ax):
        """Confusion matrix media"""
        stats = self.results['aggregate_stats']
        
        cm_data = np.array([
            [stats['true_negatives_mean'], stats['false_positives_mean']],
            [stats['false_negatives_mean'], stats['true_positives_mean']]
        ])
        
        sns.heatmap(cm_data, annot=True, fmt='.1f', cmap='Blues',
                   xticklabels=['Pred 0', 'Pred 1'],
                   yticklabels=['True 0', 'True 1'], ax=ax)
        ax.set_title('Average Confusion Matrix', fontsize=14, fontweight='bold')
    
    def _plot_roc_curve(self, ax):
        """ROC Curve aggregata"""
        y_true = self.results['all_true_labels']
        y_scores = self.results['all_probabilities']
        
        fpr, tpr, _ = roc_curve(y_true, y_scores)
        auc_score = roc_auc_score(y_true, y_scores)
        
        ax.plot(fpr, tpr, color=self.colors['primary'], linewidth=3,
               label=f'ROC Curve (AUC = {auc_score:.3f})')
        ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random')
        
        ax.set_xlabel('False Positive Rate')
        ax.set_ylabel('True Positive Rate')
        ax.set_title('ROC Curve', fontsize=14, fontweight='bold')
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    def _plot_precision_recall_curve(self, ax):
        """Precision-Recall Curve"""
        y_true = self.results['all_true_labels']
        y_scores = self.results['all_probabilities']
        
        precision, recall, _ = precision_recall_curve(y_true, y_scores)
        ap_score = average_precision_score(y_true, y_scores)
        
        ax.plot(recall, precision, color=self.colors['secondary'], linewidth=3,
               label=f'PR Curve (AP = {ap_score:.3f})')
        
        baseline = np.sum(y_true) / len(y_true)
        ax.axhline(y=baseline, color='k', linestyle='--', alpha=0.5,
                  label=f'Baseline = {baseline:.3f}')
        
        ax.set_xlabel('Recall')
        ax.set_ylabel('Precision')
        ax.set_title('Precision-Recall Curve', fontsize=14, fontweight='bold')
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    def _plot_probability_distribution(self, ax):
        """Distribuzione delle probabilità predette"""
        y_true = self.results['all_true_labels']
        y_scores = self.results['all_probabilities']
        
        ax.hist(y_scores[y_true == 0], bins=30, alpha=0.7, 
               color=self.colors['accent'], label='Class 0', density=True)
        ax.hist(y_scores[y_true == 1], bins=30, alpha=0.7,
               color=self.colors['primary'], label='Class 1', density=True)
        
        ax.set_xlabel('Predicted Probability')
        ax.set_ylabel('Density')
        ax.set_title('Probability Distribution by True Class', 
                    fontsize=14, fontweight='bold')
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    def _plot_metrics_stability(self, ax):
        """Stabilità delle metriche across splits"""
        df = self.results['individual_results']
        
        metrics = ['accuracy', 'auc_roc', 'f1']
        colors = [self.colors['primary'], self.colors['secondary'], self.colors['accent']]
        
        for metric, color in zip(metrics, colors):
            ax.plot(df['split_id'], df[metric], 'o-', color=color, 
                   label=metric.upper(), linewidth=2, markersize=6)
        
        ax.set_xlabel('Test Split')
        ax.set_ylabel('Score')
        ax.set_title('Metrics Stability Across Splits', 
                    fontsize=14, fontweight='bold')
        ax.legend()
        ax.grid(True, alpha=0.3)
        ax.set_ylim(0, 1.05)
    
    def _plot_accuracy_vs_auc(self, ax):
        """Scatter Accuracy vs AUC"""
        df = self.results['individual_results']
        
        scatter = ax.scatter(df['accuracy'], df['auc_roc'], 
                            c=df['f1'], cmap='viridis', 
                            s=100, alpha=0.7, edgecolors='black')
        
        ax.set_xlabel('Accuracy')
        ax.set_ylabel('AUC-ROC')
        ax.set_title('Accuracy vs AUC-ROC\n(colored by F1-score)', 
                    fontsize=14, fontweight='bold')
        
        # Linea di trend
        z = np.polyfit(df['accuracy'], df['auc_roc'], 1)
        p = np.poly1d(z)
        ax.plot(df['accuracy'], p(df['accuracy']), "r--", alpha=0.8)
        
        plt.colorbar(scatter, ax=ax, label='F1-Score')
        ax.grid(True, alpha=0.3)
    
    def _plot_summary_table(self, ax):
        """Tabella riassuntiva delle metriche"""
        stats = self.results['aggregate_stats']
        
        # Dati per la tabella
        table_data = [
            ['Accuracy', f"{stats['accuracy_mean']:.3f} ± {stats['accuracy_std']:.3f}"],
            ['Precision', f"{stats['precision_mean']:.3f} ± {stats['precision_std']:.3f}"],
            ['Recall', f"{stats['recall_mean']:.3f} ± {stats['recall_std']:.3f}"],
            ['F1-Score', f"{stats['f1_mean']:.3f} ± {stats['f1_std']:.3f}"],
            ['AUC-ROC', f"{stats['auc_roc_mean']:.3f} ± {stats['auc_roc_std']:.3f}"],
            ['AUC-PR', f"{stats['auc_pr_mean']:.3f} ± {stats['auc_pr_std']:.3f}"],
            ['MCC', f"{stats['mcc_mean']:.3f} ± {stats['mcc_std']:.3f}"]
        ]
        
        table = ax.table(cellText=table_data,
                        colLabels=['Metric', 'Mean ± Std'],
                        cellLoc='center',
                        loc='center',
                        colWidths=[0.4, 0.6])
        
        table.auto_set_font_size(False)
        table.set_fontsize(10)
        table.scale(1, 2)
        
        # Styling della tabella
        for i in range(len(table_data) + 1):
            for j in range(2):
                if i == 0:  # Header
                    table[(i, j)].set_facecolor(self.colors['primary'])
                    table[(i, j)].set_text_props(weight='bold', color='white')
                else:
                    if i % 2 == 0:
                        table[(i, j)].set_facecolor(self.colors['light'])
        
        ax.set_title('Performance Summary', fontsize=14, fontweight='bold')
        ax.axis('off')
    
    def export_results_to_excel(self, filename='xgboost_deeptest_results.xlsx'):
        """
        Esporta tutti i risultati in Excel
        """
        if not self.results:
            print("❌ Esegui prima run_comprehensive_test()")
            return
        
        with pd.ExcelWriter(filename, engine='openpyxl') as writer:
            # Sheet 1: Risultati individuali
            self.results['individual_results'].to_excel(
                writer, sheet_name='Individual_Results', index=False)
            
            # Sheet 2: Statistiche aggregate
            agg_df = pd.DataFrame.from_dict(
                self.results['aggregate_stats'], orient='index', 
                columns=['Value']).reset_index()
            agg_df.rename(columns={'index': 'Metric'}, inplace=True)
            agg_df.to_excel(writer, sheet_name='Aggregate_Stats', index=False)
            
            # Sheet 3: Predizioni complete
            pred_df = pd.DataFrame({
                'true_labels': self.results['all_true_labels'],
                'predictions': self.results['all_predictions'], 
                'probabilities': self.results['all_probabilities']
            })
            pred_df.to_excel(writer, sheet_name='All_Predictions', index=False)
        
        print(f"✅ Risultati XGBoost esportati in: {filename}")
    


In [ ]:
"""test_data_with_features_df = pd.read_csv(r"C:/Users/atogni/Desktop/rongowai/temp_data/geoq/test_features_extracted_enh_enc_1M.csv")
test_labels_clean_df = pd.read_csv(r"C:/Users/atogni/Desktop/rongowai/temp_data/geoq/test_labels_extracted_enh_enc_1M.csv")
test_labels_clean = test_labels_clean_df['0'].values"""

In [ ]:
scaler = joblib.load(scaler_path)
test_data_scaled = scaler.transform(test_data_with_features_df)

In [ ]:
tester = DeepTest(
    model_path=final_model_path,
    test_data=pd.DataFrame(test_data_scaled),
    test_labels=test_labels_clean_df['0'].values
)

tester.load_model(model_format='pickle')  # oppure 'json' o 'xgb_native'


results = tester.run_comprehensive_test(n_splits=10, verbose=True)

tester.print_detailed_report()

tester.create_professional_visualizations(save_path=f"{os.environ['ML_FLOW_EXPERIMENT_NAME']}_xgboost_analysis.png")


feature_imp = tester.get_feature_importance(importance_type='gain')


tester.export_results_to_excel(f"{os.environ['ML_FLOW_EXPERIMENT_NAME']}_xgboost_test_results.xlsx")
